# Step 4.5. Clustering Visualization / Diagnostics

## 목적
Step 4에서 생성한 review shortlist 후보를 같은 기준으로 시각화하여 최종 segment 구조를 진단한다.

## 입력 파일
- `data/analysis/clustering_results/05_all_candidate_cluster_labels.csv`
- `data/analysis/clustering_results/07_review_shortlist_config.json`
- `data/analysis/clustering_results/11_df_cluster_input_used.csv`
- `data/analysis/clustering_results/12_df_profile_table_used.csv`

## 출력 파일
- `data/analysis/clustering_visualizations/core_7_KMeans_k4/`
- `data/analysis/clustering_visualizations/core_7_KMeans_k5/`
- `data/analysis/clustering_visualizations/core_7_KMeans_k6/`
- `data/analysis/clustering_visualizations/_final_selected_model_core7_kmeans_k5/`

## 전체 프로젝트에서의 역할
이 단계는 Step 4의 후보 모델을 시각적으로 점검하고, Step 5에서 사용할 최종 k=5 해석 자료를 준비한다.

## 주요 분석 로직
Cluster size, profile heatmap, feature separation, 이탈률, high-risk signal, PCA/LDA projection을 후보별로 저장한다.

## 결과 해석 시 주의점
Scatter plot은 진단 자료일 뿐 최종 선택의 유일한 기준이 아니다. 실제 business interpretation은 Step 5의 profile heatmap, 이탈률, high-risk signal, strategy table을 중심으로 수행한다.

## 0. 실행 전 확인 사항

Step 4를 먼저 실행하여 아래 파일이 존재해야 한다.

```text
data/analysis/clustering_results/
├── 05_all_candidate_cluster_labels.csv
├── 07_review_shortlist_config.json
├── 11_df_cluster_input_used.csv
└── 12_df_profile_table_used.csv
```

만약 `07_review_shortlist_config.json`이 없다면, 이 노트북은 기본값으로 아래 후보를 사용한다.

```text
core_7_KMeans_k4
core_7_KMeans_k5
core_7_KMeans_k6
```

In [1]:
# ============================================================
# 0. Imports and project setup
# ============================================================

from pathlib import Path
import sys
import json
import os
import random
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Reproducibility settings
# ------------------------------------------------------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# ------------------------------------------------------------
# Project root finder
# ------------------------------------------------------------
def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current] + list(current.parents)

    for path in candidates:
        if (path / "src" / "common.py").exists():
            return path

    return current


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))


# ------------------------------------------------------------
# Common functions
# ------------------------------------------------------------
try:
    from src.common import (
        read_csv_safely,
        configure_korean_font,
        save_csv,
        save_json,
        get_plotly_layout,
    )
except ImportError:
    # fallback: src/common.py가 없거나 함수명이 다른 경우를 대비
    def read_csv_safely(path, **kwargs):
        for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
            try:
                return pd.read_csv(path, encoding=enc, **kwargs)
            except UnicodeDecodeError:
                continue
        return pd.read_csv(path, **kwargs)

    def configure_korean_font(verbose=True):
        import matplotlib
        from matplotlib import font_manager

        candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]
        installed = {font.name for font in font_manager.fontManager.ttflist}
        selected = next((f for f in candidates if f in installed), None)

        if selected is not None:
            plt.rcParams["font.family"] = selected
        plt.rcParams["axes.unicode_minus"] = False

        if verbose:
            print("Selected matplotlib font:", selected)

    def save_csv(df, path, index=False):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(path, index=index, encoding="utf-8-sig")

    def save_json(obj, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)

    def get_plotly_layout(title=None):
        return {
            "title": title,
            "template": "plotly_white",
            "font": {"family": "Arial"},
        }


configure_korean_font(verbose=False)

print("Project root:", ROOT)
print("RANDOM_STATE:", RANDOM_STATE)


Project root: C:\Users\user\Desktop\BankChurners-Customer-Segmentation
RANDOM_STATE: 42


## 1. Step 4 결과 로드

Step 4에서 저장한 candidate label, model selection score, clustering input, profile table을 불러온다.  
`review shortlist`가 저장되어 있으면 해당 후보를 사용하고, 없으면 `core_7_KMeans_k4~k6`을 기본 비교 대상으로 사용한다.

In [2]:
# ============================================================
# 1. Load Step 4 outputs
# ============================================================

input_dir = ROOT / "data" / "analysis" / "clustering_results"
output_dir = ROOT / "data" / "analysis" / "clustering_visualizations"
output_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    input_dir / "05_all_candidate_cluster_labels.csv",
    input_dir / "11_df_cluster_input_used.csv",
    input_dir / "12_df_profile_table_used.csv",
]

missing_files = [p for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Step 4 결과 파일이 부족합니다. 먼저 Step 4를 실행하세요.\n"
        + "\n".join(str(p.relative_to(ROOT)) for p in missing_files)
    )

df_cluster = read_csv_safely(input_dir / "11_df_cluster_input_used.csv")
df_profile = read_csv_safely(input_dir / "12_df_profile_table_used.csv")
all_labels_df = read_csv_safely(input_dir / "05_all_candidate_cluster_labels.csv")

# model selection score는 파일명 변경 가능성을 고려하여 탐색
score_candidates = [
    input_dir / "03_model_selection_score.csv",
    input_dir / "04_final_model_comparison_core7_kmeans_k456.csv",
    input_dir / "04_candidate_metrics_sorted.csv",
    input_dir / "04_final_model_comparison_sorted.csv",
    input_dir / "03_final_model_comparison.csv",
]

model_metrics = None
for path in score_candidates:
    if path.exists():
        model_metrics = read_csv_safely(path)
        print("Loaded model metrics:", path.relative_to(ROOT))
        break

if model_metrics is None:
    print("Model metrics file not found. Visualization will proceed without metric columns.")
    model_metrics = pd.DataFrame({"key": [c for c in all_labels_df.columns if c != "Row_ID"]})

# selected model info 로드
selected_model_path = input_dir / "09_selected_model.json"
if selected_model_path.exists():
    with open(selected_model_path, "r", encoding="utf-8") as f:
        selected_model_info = json.load(f)
    FINAL_MODEL_KEY = selected_model_info.get("selected_key", "core_7_KMeans_k5")
else:
    selected_model_info = {}
    FINAL_MODEL_KEY = "core_7_KMeans_k5"

# review shortlist config 로드
review_config_path = input_dir / "07_review_shortlist_config.json"

if review_config_path.exists():
    with open(review_config_path, "r", encoding="utf-8") as f:
        review_config = json.load(f)
    REVIEW_KEYS = review_config.get("review_keys", [
        "core_7_KMeans_k4",
        "core_7_KMeans_k5",
        "core_7_KMeans_k6",
    ])
    FINAL_MODEL_KEY = review_config.get("final_selected_key", FINAL_MODEL_KEY)
else:
    review_config = {}
    REVIEW_KEYS = [
        "core_7_KMeans_k4",
        "core_7_KMeans_k5",
        "core_7_KMeans_k6",
    ]

# 최종 모델은 review 후보에 포함되도록 보장
if FINAL_MODEL_KEY not in REVIEW_KEYS:
    REVIEW_KEYS.append(FINAL_MODEL_KEY)

missing_label_cols = [key for key in REVIEW_KEYS if key not in all_labels_df.columns]
if missing_label_cols:
    raise ValueError(f"label 파일에 후보 key가 없습니다: {missing_label_cols}")

if FINAL_MODEL_KEY not in all_labels_df.columns:
    raise ValueError(f"최종 모델 label이 없습니다: {FINAL_MODEL_KEY}")

print("Review keys:", REVIEW_KEYS)
print("Final model key:", FINAL_MODEL_KEY)


Loaded model metrics: data\analysis\clustering_results\03_model_selection_score.csv
Review keys: ['core_7_KMeans_k4', 'core_7_KMeans_k5', 'core_7_KMeans_k6']
Final model key: core_7_KMeans_k5


## 2. Feature set 정의

현재 주요 비교 대상은 `core_7`이다.  
`core_7_plus_capacity` 결과가 있을 경우에도 시각화 가능하도록 feature set을 함께 정의한다.

In [3]:
# ============================================================
# 2. Feature set definitions
# ============================================================

FEATURE_SETS = {
    "core_7": [
        "Activity_Score",
        "Count_Decline_Risk_Score",
        "Amount_Decline_Risk_Score",
        "Inactivity_Score",
        "Product_Relationship_Score",
        "Contact_Burden_Score",
        "Credit_Utilization_Score",
    ],
    "core_7_plus_capacity": [
        "Activity_Score",
        "Count_Decline_Risk_Score",
        "Amount_Decline_Risk_Score",
        "Inactivity_Score",
        "Product_Relationship_Score",
        "Contact_Burden_Score",
        "Credit_Utilization_Score",
        "Credit_Capacity_Score",
    ],
}

def infer_feature_set_from_key(key: str) -> str:
    if key.startswith("core_7_plus_capacity"):
        return "core_7_plus_capacity"
    if key.startswith("core_7"):
        return "core_7"
    raise ValueError(f"feature set을 key에서 추론할 수 없습니다: {key}")


for key in REVIEW_KEYS:
    feature_set_name = infer_feature_set_from_key(key)
    missing_features = [col for col in FEATURE_SETS[feature_set_name] if col not in df_cluster.columns]
    if missing_features:
        raise ValueError(f"{key} 시각화에 필요한 feature가 df_cluster에 없습니다: {missing_features}")

print("Feature set check completed.")

Feature set check completed.


## 3. 진단 시각화 함수 정의

각 candidate key에 대해 다음 결과를 생성한다.

- cluster size summary
- cluster profile heatmap
- feature separation summary
- attrition / high-risk signal summary
- 2D/3D PCA scatter
- direct 3D views
- 3D LDA projection

In [4]:
# ============================================================
# 3. Helper functions
# ============================================================

def safe_hover_cols(df: pd.DataFrame, cols: list[str]) -> list[str]:
    return [col for col in cols if col in df.columns]


def cluster_label_series(labels: np.ndarray) -> pd.Series:
    return pd.Series(labels).map(lambda x: f"C{int(x)}")


def save_matplotlib_fig(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


def bar_annotate(ax, bars, values, fmt="{:.1f}"):
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            fmt.format(value),
            ha="center",
            va="bottom",
            fontsize=9,
        )

In [5]:
# ============================================================
# 4. Candidate visualization function
# ============================================================

def visualize_candidate(candidate_key: str) -> dict:
    feature_set_name = infer_feature_set_from_key(candidate_key)
    feature_cols = FEATURE_SETS[feature_set_name]

    candidate_dir = output_dir / candidate_key
    candidate_dir.mkdir(parents=True, exist_ok=True)

    labels = all_labels_df[candidate_key].values

    vis_df = df_cluster[["Row_ID"] + feature_cols].copy()
    vis_df["Cluster"] = labels
    vis_df["Cluster_Label"] = cluster_label_series(labels)

    # profiling 변수 연결
    attach_cols = [
        "Attrition_Flag",
        "Attrition_Label",
        "High_Risk_Signal_Count",
        "Strong_Decline_Flag",
        "Decline_Low_Activity_Flag",
        "High_Inactive_Flag",
        "High_Contact_Flag",
        "Low_Bank_Product_Count_Flag",
        "Zero_Utilization_Flag",
        "Total_Trans_Ct",
        "Total_Trans_Amt",
        "Total_Ct_Chng_Q4_Q1",
        "Total_Amt_Chng_Q4_Q1",
        "Months_Inactive_12_mon",
        "Contacts_Count_12_mon",
        "Total_Relationship_Count",
        "Avg_Utilization_Ratio",
    ]

    for col in attach_cols:
        if col in df_profile.columns and col not in vis_df.columns:
            vis_df[col] = df_profile[col].values

    save_csv(vis_df, candidate_dir / "visualization_table.csv")

    # --------------------------------------------------------
    # 1. Cluster size summary
    # --------------------------------------------------------
    size_summary = (
        vis_df.groupby(["Cluster", "Cluster_Label"])
        .size()
        .reset_index(name="n_customers")
        .sort_values("Cluster")
    )
    size_summary["cluster_ratio"] = size_summary["n_customers"] / len(vis_df)
    save_csv(size_summary, candidate_dir / "01_cluster_size_summary.csv")

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(size_summary["Cluster_Label"], size_summary["cluster_ratio"] * 100)
    ax.set_title(f"Cluster Size Ratio - {candidate_key}")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Customer Ratio (%)")
    bar_annotate(ax, bars, size_summary["cluster_ratio"] * 100, fmt="{:.1f}%")
    save_matplotlib_fig(candidate_dir / "01_cluster_size_ratio.png")

    # --------------------------------------------------------
    # 2. Cluster profile heatmap
    # --------------------------------------------------------
    axis_profile = (
        vis_df.groupby("Cluster_Label")[feature_cols]
        .mean()
        .reindex([f"C{i}" for i in sorted(np.unique(labels))])
    )
    save_csv(axis_profile.reset_index(), candidate_dir / "02_cluster_axis_profile_mean.csv")

    heatmap_data = axis_profile.values

    fig, ax = plt.subplots(figsize=(12, max(4.5, 0.75 * len(axis_profile))))
    im = ax.imshow(heatmap_data, aspect="auto")
    plt.colorbar(im, ax=ax, label="Standardized Mean")

    ax.set_xticks(np.arange(len(feature_cols)))
    ax.set_xticklabels(feature_cols, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(axis_profile.index)))
    ax.set_yticklabels(axis_profile.index)

    ax.set_title(f"Cluster Profile Heatmap - {candidate_key}")
    ax.set_xlabel("Behavior Axis")
    ax.set_ylabel("Cluster")

    for i in range(heatmap_data.shape[0]):
        for j in range(heatmap_data.shape[1]):
            ax.text(j, i, f"{heatmap_data[i, j]:.2f}", ha="center", va="center", fontsize=8)

    save_matplotlib_fig(candidate_dir / "02_cluster_profile_heatmap.png")

    # feature separation summary
    feature_range = axis_profile.max(axis=0) - axis_profile.min(axis=0)
    feature_abs_mean = axis_profile.abs().mean(axis=0)

    feature_separation = pd.DataFrame({
        "feature": feature_cols,
        "cluster_mean_range": feature_range.values,
        "mean_abs_cluster_mean": feature_abs_mean.values,
    }).sort_values("cluster_mean_range", ascending=False)

    save_csv(feature_separation, candidate_dir / "03_feature_separation_summary.csv")

    # --------------------------------------------------------
    # 3. Attrition and high-risk signal summaries
    # --------------------------------------------------------
    attrition_summary = None

    if "Attrition_Flag" in vis_df.columns:
        agg_dict = {
            "n_customers": ("Cluster", "count"),
            "attrition_rate": ("Attrition_Flag", "mean"),
        }

        if "High_Risk_Signal_Count" in vis_df.columns:
            agg_dict["high_risk_signal_mean"] = ("High_Risk_Signal_Count", "mean")

        attrition_summary = (
            vis_df.groupby(["Cluster", "Cluster_Label"])
            .agg(**agg_dict)
            .reset_index()
            .sort_values("Cluster")
        )

        attrition_summary["cluster_ratio"] = attrition_summary["n_customers"] / len(vis_df)
        save_csv(attrition_summary, candidate_dir / "04_attrition_and_risk_summary.csv")

        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar(attrition_summary["Cluster_Label"], attrition_summary["attrition_rate"] * 100)
        overall_attrition = df_profile["Attrition_Flag"].mean() * 100

        ax.axhline(
            overall_attrition,
            linestyle="--",
            linewidth=1,
            label=f"Overall: {overall_attrition:.1f}%"
        )

        ax.set_title(f"Attrition Rate by Cluster - {candidate_key}")
        ax.set_xlabel("Cluster")
        ax.set_ylabel("Attrition Rate (%)")
        ax.legend()
        bar_annotate(ax, bars, attrition_summary["attrition_rate"] * 100, fmt="{:.1f}%")
        save_matplotlib_fig(candidate_dir / "04_attrition_rate_by_cluster.png")

    if "High_Risk_Signal_Count" in vis_df.columns:
        risk_summary = (
            vis_df.groupby(["Cluster", "Cluster_Label"])["High_Risk_Signal_Count"]
            .mean()
            .reset_index(name="high_risk_signal_mean")
            .sort_values("Cluster")
        )

        save_csv(risk_summary, candidate_dir / "05_high_risk_signal_summary.csv")

        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar(risk_summary["Cluster_Label"], risk_summary["high_risk_signal_mean"])
        ax.set_title(f"High-risk Signal Count by Cluster - {candidate_key}")
        ax.set_xlabel("Cluster")
        ax.set_ylabel("Mean High-risk Signal Count")
        bar_annotate(ax, bars, risk_summary["high_risk_signal_mean"], fmt="{:.2f}")
        save_matplotlib_fig(candidate_dir / "05_high_risk_signal_by_cluster.png")

    # --------------------------------------------------------
    # 4. 2D PCA scatter
    # --------------------------------------------------------
    X = vis_df[feature_cols].copy()

    pca_2d = PCA(n_components=2, random_state=42)
    coords_2d = pca_2d.fit_transform(X)

    pca_2d_df = vis_df.copy()
    pca_2d_df["PC1"] = coords_2d[:, 0]
    pca_2d_df["PC2"] = coords_2d[:, 1]

    hover_cols = safe_hover_cols(
        pca_2d_df,
        [
            "Row_ID",
            "Cluster_Label",
            "Attrition_Label",
            "High_Risk_Signal_Count",
            "Total_Trans_Ct",
            "Total_Trans_Amt",
            "Months_Inactive_12_mon",
            "Contacts_Count_12_mon",
        ],
    )

    fig = px.scatter(
        pca_2d_df,
        x="PC1",
        y="PC2",
        color="Cluster_Label",
        hover_data=hover_cols,
        title=(
            f"2D PCA Scatter - {candidate_key} "
            f"(Explained: {pca_2d.explained_variance_ratio_.sum():.1%})"
        ),
    )
    fig.update_traces(marker=dict(size=5, opacity=0.65))
    fig.update_layout(**get_plotly_layout())
    fig.write_html(candidate_dir / "06_2d_pca_scatter.html")
    fig.show()

    # --------------------------------------------------------
    # 5. 3D PCA scatter
    # --------------------------------------------------------
    pca_3d = PCA(n_components=3, random_state=42)
    coords_3d = pca_3d.fit_transform(X)

    pca_3d_df = vis_df.copy()
    pca_3d_df["PC1"] = coords_3d[:, 0]
    pca_3d_df["PC2"] = coords_3d[:, 1]
    pca_3d_df["PC3"] = coords_3d[:, 2]

    hover_cols = safe_hover_cols(
        pca_3d_df,
        [
            "Row_ID",
            "Cluster_Label",
            "Attrition_Label",
            "High_Risk_Signal_Count",
        ],
    )

    fig = px.scatter_3d(
        pca_3d_df,
        x="PC1",
        y="PC2",
        z="PC3",
        color="Cluster_Label",
        hover_data=hover_cols,
        title=(
            f"3D PCA Scatter - {candidate_key} "
            f"(Explained: {pca_3d.explained_variance_ratio_.sum():.1%})"
        ),
    )
    fig.update_traces(marker=dict(size=3, opacity=0.65))
    fig.update_layout(**get_plotly_layout())
    fig.write_html(candidate_dir / "07_3d_pca_scatter.html")
    fig.show()

    save_json(
        {
            "feature_columns": feature_cols,
            "pca_2d_explained_variance_ratio": pca_2d.explained_variance_ratio_.tolist(),
            "pca_2d_cumulative_explained_variance": float(pca_2d.explained_variance_ratio_.sum()),
            "pca_3d_explained_variance_ratio": pca_3d.explained_variance_ratio_.tolist(),
            "pca_3d_cumulative_explained_variance": float(pca_3d.explained_variance_ratio_.sum()),
        },
        candidate_dir / "07_pca_info.json",
    )

    # --------------------------------------------------------
    # 6. Direct 3D Risk View
    # --------------------------------------------------------
    direct_risk_axes = [
        "Activity_Score",
        "Count_Decline_Risk_Score",
        "Inactivity_Score",
    ]

    if all(col in vis_df.columns for col in direct_risk_axes):
        hover_cols = safe_hover_cols(
            vis_df,
            [
                "Row_ID",
                "Cluster_Label",
                "Attrition_Label",
                "High_Risk_Signal_Count",
                "Total_Trans_Ct",
                "Months_Inactive_12_mon",
            ],
        )

        fig = px.scatter_3d(
            vis_df,
            x="Activity_Score",
            y="Count_Decline_Risk_Score",
            z="Inactivity_Score",
            color="Cluster_Label",
            hover_data=hover_cols,
            title=f"3D Direct Risk View - {candidate_key}",
            labels={
                "Activity_Score": "Activity",
                "Count_Decline_Risk_Score": "Count Decline Risk",
                "Inactivity_Score": "Inactivity",
            },
        )
        fig.update_traces(marker=dict(size=3, opacity=0.65))
        fig.update_layout(**get_plotly_layout())
        fig.write_html(candidate_dir / "08_3d_direct_risk_view.html")
        fig.show()

    # --------------------------------------------------------
    # 7. Direct 3D Behavior View
    # --------------------------------------------------------
    direct_behavior_axes = [
        "Activity_Score",
        "Contact_Burden_Score",
        "Credit_Utilization_Score",
    ]

    if all(col in vis_df.columns for col in direct_behavior_axes):
        hover_cols = safe_hover_cols(
            vis_df,
            [
                "Row_ID",
                "Cluster_Label",
                "Attrition_Label",
                "High_Risk_Signal_Count",
                "Contacts_Count_12_mon",
                "Avg_Utilization_Ratio",
            ],
        )

        fig = px.scatter_3d(
            vis_df,
            x="Activity_Score",
            y="Contact_Burden_Score",
            z="Credit_Utilization_Score",
            color="Cluster_Label",
            hover_data=hover_cols,
            title=f"3D Direct Behavior View - {candidate_key}",
            labels={
                "Activity_Score": "Activity",
                "Contact_Burden_Score": "Contact Burden",
                "Credit_Utilization_Score": "Credit Utilization",
            },
        )
        fig.update_traces(marker=dict(size=3, opacity=0.65))
        fig.update_layout(**get_plotly_layout())
        fig.write_html(candidate_dir / "09_3d_direct_behavior_view.html")
        fig.show()

    # --------------------------------------------------------
    # 8. 3D LDA projection
    # --------------------------------------------------------
    n_clusters = len(np.unique(labels))
    max_lda_components = min(3, len(feature_cols), n_clusters - 1)

    if max_lda_components == 3:
        lda = LinearDiscriminantAnalysis(n_components=3)
        coords_lda = lda.fit_transform(X, labels)

        lda_df = vis_df.copy()
        lda_df["LD1"] = coords_lda[:, 0]
        lda_df["LD2"] = coords_lda[:, 1]
        lda_df["LD3"] = coords_lda[:, 2]

        hover_cols = safe_hover_cols(
            lda_df,
            [
                "Row_ID",
                "Cluster_Label",
                "Attrition_Label",
                "High_Risk_Signal_Count",
            ],
        )

        fig = px.scatter_3d(
            lda_df,
            x="LD1",
            y="LD2",
            z="LD3",
            color="Cluster_Label",
            hover_data=hover_cols,
            title=f"3D LDA Projection - {candidate_key}",
        )
        fig.update_traces(marker=dict(size=3, opacity=0.65))
        fig.update_layout(**get_plotly_layout())
        fig.write_html(candidate_dir / "10_3d_lda_projection.html")
        fig.show()

        save_json(
            {
                "explained_variance_ratio": lda.explained_variance_ratio_.tolist(),
                "note": (
                    "LDA uses cluster labels to emphasize between-cluster differences. "
                    "It is a supervised visualization of derived clusters, not evidence "
                    "that natural clusters are perfectly separated."
                ),
            },
            candidate_dir / "10_lda_info.json",
        )

    # --------------------------------------------------------
    # 9. Candidate-level diagnostic summary
    # --------------------------------------------------------
    metric_row = model_metrics[model_metrics["key"] == candidate_key]
    metric_dict = metric_row.iloc[0].to_dict() if len(metric_row) > 0 else {}

    diagnostic = {
        "key": candidate_key,
        "feature_set": feature_set_name,
        "n_clusters": int(n_clusters),
        "min_cluster_ratio": float(size_summary["cluster_ratio"].min()),
        "max_cluster_ratio": float(size_summary["cluster_ratio"].max()),
        "mean_feature_range": float(feature_separation["cluster_mean_range"].mean()),
        "max_feature_range": float(feature_separation["cluster_mean_range"].max()),
        "mean_abs_cluster_mean": float(feature_separation["mean_abs_cluster_mean"].mean()),
        "pca_2d_explained_variance": float(pca_2d.explained_variance_ratio_.sum()),
        "pca_3d_explained_variance": float(pca_3d.explained_variance_ratio_.sum()),
    }

    if attrition_summary is not None:
        diagnostic["attrition_rate_range"] = float(
            attrition_summary["attrition_rate"].max()
            - attrition_summary["attrition_rate"].min()
        )

    if "High_Risk_Signal_Count" in vis_df.columns:
        risk_by_cluster = vis_df.groupby("Cluster")["High_Risk_Signal_Count"].mean()
        diagnostic["high_risk_signal_range"] = float(risk_by_cluster.max() - risk_by_cluster.min())

    # metric 정보 병합
    for col in [
        "silhouette",
        "davies_bouldin",
        "calinski_harabasz",
        "internal_rank_score",
        "business_rank_score",
        "overall_rank_score",
        "selection_score",
    ]:
        if col in metric_dict:
            diagnostic[col] = metric_dict[col]

    save_json(diagnostic, candidate_dir / "00_candidate_diagnostic_summary.json")

    return diagnostic

## 4. Review shortlist 전체 실행

`core_7_KMeans_k4`, `core_7_KMeans_k5`, `core_7_KMeans_k6`에 대해 같은 진단 시각화를 반복 생성한다.  
생성된 결과는 candidate별 하위 폴더에 저장된다.

In [ ]:
# ============================================================
# 5. Run visual diagnostics for all review candidates
# ============================================================

diagnostics = []

for key in REVIEW_KEYS:
    print("=" * 100)
    print(f"Visualizing: {key}")
    diagnostic = visualize_candidate(key)
    diagnostics.append(diagnostic)

diagnostics_df = pd.DataFrame(diagnostics)

sort_cols = []
ascending = []

if "internal_rank_score" in diagnostics_df.columns:
    sort_cols.append("internal_rank_score")
    ascending.append(True)

sort_cols.extend(["mean_feature_range", "attrition_rate_range"])
ascending.extend([False, False])

sort_cols = [col for col in sort_cols if col in diagnostics_df.columns]
ascending = ascending[:len(sort_cols)]

diagnostics_df_sorted = diagnostics_df.sort_values(sort_cols, ascending=ascending) if sort_cols else diagnostics_df

diagnostics_df_sorted["is_final_selected_model"] = diagnostics_df_sorted["key"].eq(FINAL_MODEL_KEY)

save_csv(
    diagnostics_df_sorted,
    output_dir / "00_review_shortlist_visual_diagnostics_summary.csv"
)

# 최종 선택 모델 k=5만 별도 요약 저장
final_diagnostic_df = diagnostics_df_sorted[diagnostics_df_sorted["key"].eq(FINAL_MODEL_KEY)].copy()
save_csv(
    final_diagnostic_df,
    output_dir / "00_final_selected_visual_diagnostic_summary.csv"
)

# 최종 보고서용 폴더 생성: 기존 candidate 폴더를 복사
final_source_dir = output_dir / FINAL_MODEL_KEY
final_report_dir = output_dir / "_final_selected_model_core7_kmeans_k5"

if final_source_dir.exists():
    if final_report_dir.exists():
        shutil.rmtree(final_report_dir)
    shutil.copytree(final_source_dir, final_report_dir)

    # 최종 모델 정보도 같이 저장
    save_json(
        {
            "final_model_key": FINAL_MODEL_KEY,
            "source_dir": str(final_source_dir.relative_to(ROOT)),
            "final_report_dir": str(final_report_dir.relative_to(ROOT)),
            "selected_model_info": selected_model_info,
            "note": (
                "이 파일은 최종 선택 모델 시각화의 원본 정보를 담고 있다. "
                "원본 상세 결과는 candidate key 폴더에 보관한다."
            ),
        },
        final_report_dir / "00_final_visualization_info.json",
    )
else:
    print(f"Final source directory not found yet: {final_source_dir}")

display(diagnostics_df_sorted)

print("Final selected visualization folder:", final_report_dir.relative_to(ROOT))

## 5. 후보 비교 해석 가이드

최종 k 선택은 하나의 점수만으로 결정하지 않고 다음 기준을 함께 본다.

1. 내부 clustering metric이 안정적인가?
2. cluster size가 과도하게 불균형하지 않은가?
3. cluster profile heatmap에서 각 cluster의 특징이 명확한가?
4. 이탈률과 high-risk signal 차이가 실제로 나타나는가?
5. 각 cluster에 서로 다른 고객 유지 전략을 붙일 수 있는가?
6. k가 너무 작아 중요한 segment가 합쳐지거나, 너무 커서 유사한 segment가 중복되지는 않는가?

현재 최종 채택 모델은 `core_7_KMeans_k5`이다.

- `k=4`: 가장 간결하지만 고위험/저위험 세분화가 부족함
- `k=5`: 초고위험군, 초저위험군, 신용 활용군, 상품 관계군, 고활동·저관계군을 균형 있게 분리함
- `k=6`: 고위험군을 가장 날카롭게 분리하지만 해석 복잡도가 증가함


In [ ]:
# ============================================================
# 6. Review README와 실행 metadata 저장
# ============================================================

readme_text = f"""# Step 4.5 Clustering Visualization Diagnostics

이 폴더는 Step 4 review shortlist 후보의 clustering 진단 시각화 결과를 보관한다.

## 최종 선택 모델

- `{FINAL_MODEL_KEY}`

보고서와 발표 준비에 바로 참고할 수 있는 최종 모델 시각화 사본은 아래 폴더에 저장된다.

```text
_final_selected_model_core7_kmeans_k5/
```

## 검토 후보

{chr(10).join(f'- {key}' for key in REVIEW_KEYS)}

## 주요 출력 파일

각 후보별로 노트북은 다음 파일을 저장한다.

- `01_cluster_size_summary.csv`
- `01_cluster_size_ratio.png`
- `02_cluster_axis_profile_mean.csv`
- `02_cluster_profile_heatmap.png`
- `03_feature_separation_summary.csv`
- `04_attrition_and_risk_summary.csv`
- `04_attrition_rate_by_cluster.png`
- `05_high_risk_signal_summary.csv`
- `05_high_risk_signal_by_cluster.png`
- `06_2d_pca_scatter.html`
- `07_3d_pca_scatter.html`
- `08_3d_direct_risk_view.html`
- `09_3d_direct_behavior_view.html`
- `10_3d_lda_projection.html`

## 해석 주의사항

이 시각화는 후보 cluster 구조를 비교하기 위한 진단 자료이다.
Scatter plot에서 군집이 완전히 분리되어 보이는지만으로 모델을 선택하지 않는다.
최종 segment 해석과 전략 설계는 Step 5의 profile heatmap, 이탈률, high-risk signal, strategy table을 중심으로 수행한다.
이 프로젝트의 최종 segment model은 `core_7_KMeans_k5`로 고정한다.
"""
(output_dir / "README_clustering_visualizations.md").write_text(readme_text, encoding="utf-8")

run_metadata = {
    "step": "Step 4.5 Clustering Visualization / Diagnostics",
    "input_dir": str(input_dir.relative_to(ROOT)),
    "output_dir": str(output_dir.relative_to(ROOT)),
    "review_keys": REVIEW_KEYS,
    "final_model_key": FINAL_MODEL_KEY,
    "final_report_dir": str((output_dir / "_final_selected_model_core7_kmeans_k5").relative_to(ROOT)),
    "objective": "core_7 KMeans k=4,5,6 후보의 시각화 진단 결과를 비교하고 최종 k=5 모델의 보고서용 시각화 폴더를 만든다.",
}
save_json(run_metadata, output_dir / "00_visualization_run_metadata.json")

print("Files saved to:", output_dir.relative_to(ROOT))
print("Final selected model:", FINAL_MODEL_KEY)